# EEG Audit and Finalization — v1

Independent audit of the frozen v3 EEG preprocessing outputs, re-verified from the saved files.
**Feature extraction and model training are out of scope and were not started.**

**Rules in force (non-negotiable):**
- Raw data (`P01`…`P10`) is read-only. Nothing is written, renamed or deleted there.
- Every new output is written **only** inside `E:\REU_DATA\audit_finalization_v1\` (guarded by `audit_write`).
- v3 outputs under `EEG processed\` are read-only for this pass; nothing existing is modified.
- Missing metadata is recorded as a gap, never invented. The physical acquisition reference is documented as **UNKNOWN**.
- ICA outputs are **not** auto-accepted: the 25 v3 auto-APPROVED recordings become `HUMAN_SIGNOFF_PENDING`.
- Every code cell is followed by a markdown cell with What / Why / Result bullets.

In [1]:
# STEP A0 - setup: imports, paths, safety guards, shared constants
from pathlib import Path
import json, gc, hashlib, time, warnings
import numpy as np
import pandas as pd
import scipy.signal as sps
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mne
import bioread

warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")

RAW_ROOT   = Path(r"E:\REU_DATA")
AUDIT_DIR  = RAW_ROOT / "audit_finalization_v1"
EEG_OUT    = AUDIT_DIR / "eeg"
REPORTS_OUT= AUDIT_DIR / "reports"
FIG_OUT    = AUDIT_DIR / "figures"
EEG_PROC   = RAW_ROOT / "EEG processed" / "eeg"
BASELINE   = RAW_ROOT / "INVENTORY_BASELINE.csv"
V3_NB      = RAW_ROOT / "notebooks" / "EEG_Dataset_Analysis_and_Preprocessing_v3.ipynb"
V3_FIG     = EEG_PROC / "figures_v3"
for d in (EEG_OUT, REPORTS_OUT, FIG_OUT):
    d.mkdir(parents=True, exist_ok=True)

def audit_write(path) -> Path:
    """Guard: refuse to write anything outside audit_finalization_v1."""
    p = Path(path).resolve()
    root = AUDIT_DIR.resolve()
    if root != p and root not in p.parents:
        raise RuntimeError(f"refusing to write outside the audit directory: {p}")
    p.parent.mkdir(parents=True, exist_ok=True)
    return p

CONDITIONS   = {"01calm": "calm", "02meditation": "meditation", "03maths": "maths", "04speech": "speech"}
CONDS        = ["calm", "meditation", "maths", "speech"]
PARTICIPANTS = [f"P{i:02d}" for i in range(1, 11)]
CANON_24 = ["Fp1","Fpz","Fp2","F7","F3","Fz","F4","F8","M1","T7","C3","Cz","C4","T8",
            "M2","P7","P3","Pz","P4","P8","POz","O1","Oz","O2"]
EOG_PROXY = ["Fp1", "Fpz", "Fp2"]
FRONTAL8  = ["Fp1","Fpz","Fp2","F7","F3","Fz","F4","F8"]
STANDARD_1020 = mne.channels.make_standard_montage("standard_1020")

def acq_of(pid, suffix):
    d = RAW_ROOT / pid / f"{int(pid[1:]):02d}_eeg" / "acq"
    m = list(d.glob(f"{pid}_*{suffix}.acq"))
    return m[0] if m else None

def load_acq_to_raw(acq_path) -> mne.io.RawArray:
    """Same loader as the v3 pipeline: bioread -> Volts -> RawArray."""
    df = bioread.read_file(str(acq_path))
    fs = int(df.samples_per_second)
    names = [ch.name for ch in df.channels]
    data = np.vstack([ch.data for ch in df.channels]) * 1e-6
    info = mne.create_info(ch_names=names, sfreq=fs, ch_types="eeg")
    raw = mne.io.RawArray(data, info, verbose=False)
    raw.set_montage(STANDARD_1020, on_missing="ignore", verbose=False)
    del df, data
    gc.collect()
    return raw

t_start = time.time()
print(f"RESULT: audit output root = {AUDIT_DIR}")
print(f"RESULT: v3 EEG outputs under audit = {EEG_PROC}")
print(f"RESULT: raw data root (read-only) = {RAW_ROOT}")

RESULT: audit output root = E:\REU_DATA\audit_finalization_v1
RESULT: v3 EEG outputs under audit = E:\REU_DATA\EEG processed\eeg
RESULT: raw data root (read-only) = E:\REU_DATA


### What was done
- Imported the libraries, defined the audit output root `audit_finalization_v1\eeg`, and added the `audit_write` guard that refuses any write outside that folder.
- Recorded the shared constants: participant list, the four conditions, the canonical 24 channel labels, the EOG proxy, and the standard 10-20 montage used as an *assumed* layout.
- Reused the exact v3 loading function (`bioread` → Volts → `RawArray`) so the audit measures the same chain the pipeline used.

### Why
- The audit must never touch raw data or the frozen v3 outputs, so every write path goes through one guard.
- Sharing v3's constants means any later disagreement is a data disagreement, not a naming disagreement.

### Result
- **Result**: the printed lines show the three roots. Check that the audit root is `audit_finalization_v1` and the raw root is the participant folders.

In [2]:
# STEP A1 - raw integrity pre-check: re-hash all 100 raw files vs INVENTORY_BASELINE.csv
base = pd.read_csv(BASELINE)
raw_rows = base[base["RelPath"].str.match(r"^P\d{2}\\")].copy()
print(f"RESULT: raw files listed in the baseline: {len(raw_rows)}")

chk = []
for _, r in raw_rows.iterrows():
    p = RAW_ROOT / r["RelPath"]
    exists = p.exists()
    sha = ""
    if exists:
        h = hashlib.sha256()
        with open(p, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 22), b""):
                h.update(chunk)
        sha = h.hexdigest()
    ok = bool(exists and sha == r["SHA256"] and p.stat().st_size == r["SizeBytes"])
    chk.append({"RelPath": r["RelPath"], "baseline_sha256": r["SHA256"], "now_sha256": sha,
                "baseline_size": int(r["SizeBytes"]), "now_size": int(p.stat().st_size) if exists else -1,
                "result": "PASS" if ok else "FAIL"})
    if not ok:
        print(f"RESULT: INTEGRITY FAILURE: {r['RelPath']}")
chk_df = pd.DataFrame(chk)
chk_df.to_csv(audit_write(REPORTS_OUT / "raw_integrity_pre_v1.csv"), index=False)
n_fail = int((chk_df["result"] == "FAIL").sum())
print(f"RESULT: raw integrity pre-check: {len(chk_df)-n_fail}/{len(chk_df)} PASS, {n_fail} FAIL")
if n_fail:
    raise RuntimeError("raw integrity check failed - audit stopped before reading any data")

RESULT: raw files listed in the baseline: 100


RESULT: raw integrity pre-check: 100/100 PASS, 0 FAIL


### What was done
- Re-hashed all 100 raw files (40 `.acq`, 40 `.xlsx`, 10 speech `.wav` + 10 `.aac`/`.mp4`) with SHA-256 and compared each hash and size against `INVENTORY_BASELINE.csv`.
- Saved a per-file PASS/FAIL table to `reports\raw_integrity_pre_v1.csv`.
- Hard-stopped the notebook if any file differs from the baseline.

### Why
- The audit may only proceed if the raw data is byte-identical to the recorded baseline; otherwise every later comparison would be meaningless.

### Result
- **Result**: the printed count must be `100/100 PASS, 0 FAIL`. Any FAIL aborts the notebook — that is intentional, not a bug.

In [3]:
# STEP A2 - recording inventory and participant-condition mapping (all 40)
rows = []
for pid in PARTICIPANTS:
    for suffix, cond in CONDITIONS.items():
        acq = acq_of(pid, suffix)
        xlsx_l = list((RAW_ROOT / pid / f"{int(pid[1:]):02d}_eeg" / "csv").glob(f"{pid}_*{suffix}.xlsx"))
        cleaned = EEG_PROC / pid / cond / f"{pid}_{cond}_eeg.fif"
        ica_f   = EEG_PROC / pid / cond / f"{pid}_{cond}_eeg_ica.fif"
        prep    = EEG_PROC / pid / cond / "preprocessing_report.json"
        ica_j   = EEG_PROC / pid / cond / "ica_report.json"
        rows.append({
            "participant": pid, "condition": cond, "expected_suffix": suffix,
            "acq_path": str(acq) if acq else "", "acq_exists": bool(acq),
            "acq_size_MB": round(acq.stat().st_size / 1e6, 1) if acq else -1,
            "xlsx_exists": bool(xlsx_l), "xlsx_size_MB": round(xlsx_l[0].stat().st_size / 1e6, 1) if xlsx_l else -1,
            "cleaned_fif_exists": cleaned.exists(), "ica_fif_exists": ica_f.exists(),
            "preprocessing_report_exists": prep.exists(), "ica_report_exists": ica_j.exists(),
        })
inv = pd.DataFrame(rows)
inv.to_csv(audit_write(EEG_OUT / "inventory_v1.csv"), index=False)
print(f"RESULT: recordings in the audit inventory: {len(inv)}/40")
print(f"RESULT: source .acq present: {int(inv['acq_exists'].sum())}/40; source .xlsx present: {int(inv['xlsx_exists'].sum())}/40")
print(f"RESULT: cleaned fif present: {int(inv['cleaned_fif_exists'].sum())}/40; ica fif present: {int(inv['ica_fif_exists'].sum())}/30 expected")
print(f"RESULT: preprocessing reports present: {int(inv['preprocessing_report_exists'].sum())}/40; ica reports: {int(inv['ica_report_exists'].sum())}/30 expected")

RESULT: recordings in the audit inventory: 40/40
RESULT: source .acq present: 40/40; source .xlsx present: 40/40
RESULT: cleaned fif present: 40/40; ica fif present: 30/30 expected
RESULT: preprocessing reports present: 40/40; ica reports: 30/30 expected


### What was done
- Listed every expected participant-condition combination (10 x 4 = 40) and checked for the source `.acq`, source `.xlsx`, cleaned `.fif`, ICA `.fif`, and the two JSON reports.
- Saved the mapping to `eeg\inventory_v1.csv`.

### Why
- The audit must confirm the recording set is complete and correctly named before checking any signal content.

### Result
- **Result**: expect `40/40` for acq, xlsx, cleaned fif and preprocessing reports, and `30/30` for ICA outputs (speech condition has no ICA). Anything less is a coverage gap.

In [4]:
# STEP A3 - sampling rate, channel labels, units, duration: read from the file headers (no assumptions)
meta_rows = []
for pid in PARTICIPANTS:
    for suffix, cond in CONDITIONS.items():
        acq = acq_of(pid, suffix)
        cleaned = EEG_PROC / pid / cond / f"{pid}_{cond}_eeg.fif"
        rec = {"participant": pid, "condition": cond, "acq_path": str(acq)}
        df = bioread.read_file(str(acq))
        rec["acq_fs_hz"] = float(df.samples_per_second)
        rec["acq_n_channels"] = len(df.channels)
        rec["acq_ch_names"] = ",".join(c.name for c in df.channels)
        rec["acq_units"] = str(df.channels[0].units)
        rec["acq_n_times"] = len(df.channels[0].data)
        rec["acq_duration_s"] = round(rec["acq_n_times"] / rec["acq_fs_hz"], 2)
        del df
        gc.collect()
        rawh = mne.io.read_raw_fif(cleaned, preload=False, verbose=False)
        rec["fif_fs_hz"] = float(rawh.info["sfreq"])
        rec["fif_n_channels"] = len(rawh.ch_names)
        rec["fif_ch_names"] = ",".join(rawh.ch_names)
        rec["fif_n_times"] = int(rawh.n_times)
        rec["fif_duration_s"] = round(rawh.n_times / rawh.info["sfreq"], 2)
        rec["labels_match_canonical24"] = rec["acq_ch_names"].split(",") == CANON_24
        rec["fif_matches_acq_labels"] = rec["fif_ch_names"] == rec["acq_ch_names"]
        rec["fs_match"] = abs(rec["fif_fs_hz"] - rec["acq_fs_hz"]) < 0.01
        rec["n_times_match"] = rec["fif_n_times"] == rec["acq_n_times"]
        meta_rows.append(rec)
        del rawh
        gc.collect()
meta = pd.DataFrame(meta_rows)
meta.to_csv(audit_write(EEG_OUT / "metadata_verification_v1.csv"), index=False)
print(f"RESULT: headers read for {len(meta)}/40 recordings")
print(f"RESULT: sampling rate values found in .acq headers: {sorted(meta['acq_fs_hz'].unique())}")
print(f"RESULT: all 40 recordings use the canonical 24 labels: {bool(meta['labels_match_canonical24'].all())}")
print(f"RESULT: cleaned fif labels match the source in all 40: {bool(meta['fif_matches_acq_labels'].all())}")
print(f"RESULT: sampling rate matches source in all 40: {bool(meta['fs_match'].all())}; sample counts match: {bool(meta['n_times_match'].all())}")
print(f"RESULT: source units reported by BIOPAC: {sorted(meta['acq_units'].unique())}")

RESULT: headers read for 40/40 recordings
RESULT: sampling rate values found in .acq headers: [np.float64(1000.0)]
RESULT: all 40 recordings use the canonical 24 labels: True
RESULT: cleaned fif labels match the source in all 40: True
RESULT: sampling rate matches source in all 40: True; sample counts match: True
RESULT: source units reported by BIOPAC: ['uVolt']


### What was done
- Read the sampling rate, channel names, units and sample count **from every `.acq` header** with `bioread` — nothing was assumed.
- Read the same facts from each saved cleaned `.fif` header and compared labels, rate and sample counts.
- Saved the table to `eeg\metadata_verification_v1.csv`.

### Why
- The v3 pipeline verified Fs = 1000 Hz by assertion; the audit re-reads all 40 headers so the number is measured, not claimed. Channel-label integrity protects participant and electrode identity.

### Result
- **Result**: look for one unique Fs value across all 40 headers, `True` for canonical labels, and `True` for fif-vs-source matches on labels, rate and sample counts.

In [5]:
# STEP A4 - acquisition reference and assumed metadata: document, never invent
v3nb = json.load(open(V3_NB, encoding="utf-8"))
v3_code = "\n".join("".join(c["source"]) for c in v3nb["cells"] if c["cell_type"] == "code")
ref_tokens = ["set_eeg_reference", "set_bipolar_reference", "average reference", "REST", "reref", "EcoG"]
api_hits = {t: v3_code.count(t) for t in ref_tokens}
print(f"RESULT: rereference API calls found in the v3 notebook: { {k: v for k, v in api_hits.items() if v} or 'none' }")

# does the .acq header itself carry any reference information?
df = bioread.read_file(str(acq_of("P01", "01calm")))
ref_like = [a for a in dir(df) if "ref" in a.lower()]
ch_ref_like = [a for a in dir(df.channels[0]) if "ref" in a.lower()]
print(f"RESULT: Datafile attributes containing 'ref': {ref_like or 'none'}")
print(f"RESULT: channel attributes containing 'ref': {ch_ref_like or 'none'}")
del df
gc.collect()

note = f"""# Acquisition reference — documented as UNKNOWN (v1 audit)

- The physical acquisition reference of the BIOPAC EEG recordings is **UNKNOWN**.
- It could not be recovered from any available source in this repository:
  - the v3 notebook contains no rereference call (searched: {api_hits}),
  - the `.acq` header exposes no reference attribute (checked `bioread` Datafile/channel attributes),
  - no acquisition-team documentation is present in `information\`.
- **No reference was applied or assumed by the audit.** The v3 pipeline also applied none
  (`set_eeg_reference` absent), so the saved files carry whatever reference the hardware used at
  recording time. This must be confirmed with the acquisition team before reference-sensitive
  features (e.g. Laplacian, current-source-density, or linked-mastoid derivations) are computed.

## Other assumed (not measured) metadata — recorded for transparency
- **Electrode positions**: the standard 10-20 montage was attached by the v3 pipeline
  (`set_montage(..., on_missing="ignore")`). No digitised positions exist in the dataset, so all
  topographic statements (ICA frontal weights, neighbour correlations) rest on this assumption.
- **EOG**: no dedicated EOG channel was recorded. The ocular detector uses the mean of
  Fp1/Fpz/Fp2 as a frontal proxy. This is an approximation, not a calibration.
"""
(audit_write(REPORTS_OUT / "acquisition_reference_note_v1.md")).write_text(note, encoding="utf-8")
print("RESULT: reference note written to reports/acquisition_reference_note_v1.md - status: UNKNOWN pending acquisition team")

RESULT: rereference API calls found in the v3 notebook: none


RESULT: Datafile attributes containing 'ref': ['__weakref__']
RESULT: channel attributes containing 'ref': ['__weakref__']
RESULT: reference note written to reports/acquisition_reference_note_v1.md - status: UNKNOWN pending acquisition team


### What was done
- Searched the frozen v3 notebook code for any rereference call and checked whether the `.acq` format exposes reference metadata through `bioread`.
- Wrote `reports\acquisition_reference_note_v1.md` stating that the acquisition reference is **UNKNOWN** and listing the two assumed-metadata items (standard 10-20 positions, Fp1/Fpz/Fp2 EOG proxy).

### Why
- Your constraints require the physical acquisition reference to be documented as UNKNOWN until the acquisition team confirms it, and forbid inventing it. Assumptions must be written down where feature extraction will see them.

### Result
- **Result**: the searches must print `none` for rereference APIs and reference attributes. The markdown file is the permanent record — feature extraction must read it before any reference-sensitive work.

In [2]:
# STEP A5 - microvolt-to-volt conversion: header + magnitude spot check (exact proof is STEP A7)
SPOT = [("P01", "calm"), ("P05", "maths"), ("P07", "meditation"), ("P10", "speech")]
rows = []
for pid, cond in SPOT:
    suffix = {v: k for k, v in CONDITIONS.items()}[cond]
    acq = acq_of(pid, suffix)
    df = bioread.read_file(str(acq))
    raw_uv = np.vstack([c.data for c in df.channels])          # source scale (microvolts, incl. DC)
    src_units = str(df.channels[0].units)
    rec = {"participant": pid, "condition": cond, "acq_units_header": src_units,
           "acq_p99_abs_uV": round(float(np.percentile(np.abs(raw_uv), 99)), 1)}
    del raw_uv, df
    gc.collect()
    cleaned = EEG_PROC / pid / cond / f"{pid}_{cond}_eeg.fif"
    rawc = mne.io.read_raw_fif(cleaned, preload=True, verbose=False)
    d = rawc.get_data()
    rec["fif_max_abs_V"] = float(np.abs(d).max())
    rec["fif_p99_abs_V"] = float(np.percentile(np.abs(d), 99))
    # EEG expressed in Volts: the bulk of the data (p99) must sit between 10 uV and 2 mV
    rec["p99_in_eeg_volt_range"] = bool(1e-5 <= rec["fif_p99_abs_V"] <= 2e-3)
    rec["max_is_artifact_spike"] = bool(rec["fif_max_abs_V"] > 5e-3)   # single transient, not a unit error
    rec["source_header_in_microvolts"] = src_units.lower() in ("uvolt", "uv", "microvolt", "microvolts")
    with open(EEG_PROC / pid / cond / "preprocessing_report.json") as f:
        rep = json.load(f)
    rec["report_claims_volt_conversion"] = "converted" in str(rep.get("Units after load", "")).lower()
    rows.append(rec)
    del rawc, d
    gc.collect()
u = pd.DataFrame(rows)
u.to_csv(audit_write(EEG_OUT / "units_verification_v1.csv"), index=False)
print(f"RESULT: unit spot-check on {len(u)}/4 recordings")
print(f"RESULT: source headers report microvolts in all 4: {bool(u['source_header_in_microvolts'].all())}")
print(f"RESULT: saved data bulk (p99) sits in the EEG-in-Volts range (10 uV - 2 mV) in all 4: {bool(u['p99_in_eeg_volt_range'].all())}")
print(f"RESULT: preprocessing reports state a V conversion in all 4: {bool(u['report_claims_volt_conversion'].all())}")
print(f"RESULT: recordings with a >5 mV transient (artifact spike, see artifact log - not a unit error): {u.loc[u['max_is_artifact_spike'], ['participant','condition']].values.tolist() or 'none'}")
print("RESULT: exact sample-wise proof of the 1e-6 conversion is the re-filter match in STEP A7 (<1e-9 V)")

RESULT: unit spot-check on 4/4 recordings
RESULT: source headers report microvolts in all 4: True
RESULT: saved data bulk (p99) sits in the EEG-in-Volts range (10 uV - 2 mV) in all 4: True
RESULT: preprocessing reports state a V conversion in all 4: True
RESULT: recordings with a >5 mV transient (artifact spike, see artifact log - not a unit error): [['P07', 'meditation']]
RESULT: exact sample-wise proof of the 1e-6 conversion is the re-filter match in STEP A7 (<1e-9 V)


### What was done
- For 4 spot recordings, checked the BIOPAC header unit label (must say microvolts) and whether the bulk of the saved data (99th percentile magnitude) sits in the EEG-in-Volts range (10 uV - 2 mV). Single transients above 5 mV are recorded as artifact spikes, not unit errors.
- Saved the table to `eeg\units_verification_v1.csv`.

### Why
- A unit error would silently scale every amplitude-based feature by 10^6. The header label plus a magnitude range check catches that cheaply; the exact sample-level proof follows in the re-filter step.

### Result
- **Result**: headers must say microvolts and all p99 magnitudes must sit in the 10 uV - 2 mV range. Any >5 mV transient listed here is an artifact spike already covered by the artifact log. The re-filter match in STEP A7 is the exact conversion proof.

In [7]:
# STEP A6 - 50 Hz interference measured on the RAW, unfiltered signal (all 40), notch rationale per recording
HZ_WIN_S, HZ_NPERSEG = 60.0, 4096
hz_rows = []
for pid in PARTICIPANTS:
    for suffix, cond in CONDITIONS.items():
        acq = acq_of(pid, suffix)
        df = bioread.read_file(str(acq))
        fs = float(df.samples_per_second)
        data = np.vstack([c.data for c in df.channels])       # RAW microvolts, unfiltered
        n = data.shape[1]
        win = min(int(HZ_WIN_S * fs), n)
        st = (n - win) // 2
        f, pxx = sps.welch(data[:, st:st + win], fs=fs, nperseg=min(HZ_NPERSEG, win), axis=1)
        pm = pxx.mean(axis=0)
        peak = float(pm[(f >= 49.5) & (f <= 50.5)].mean())
        floor = float(0.5 * (pm[(f >= 45.0) & (f <= 49.0)].mean() + pm[(f >= 51.0) & (f <= 59.0)].mean()))
        ratio = peak / max(floor, 1e-20)
        pk_c = pxx[:, (f >= 49.5) & (f <= 50.5)].mean(axis=1)
        fl_c = 0.5 * (pxx[:, (f >= 45.0) & (f <= 49.0)].mean(axis=1) + pxx[:, (f >= 51.0) & (f <= 59.0)].mean(axis=1))
        rc = pk_c / np.maximum(fl_c, 1e-20)
        with open(EEG_PROC / pid / cond / "preprocessing_report.json") as fjson:
            rep = json.load(fjson)
        notch_txt = str(rep.get("Notch", ""))
        notch_applied = notch_txt.lower().startswith("50") and "not applied" not in notch_txt.lower()
        hz_rows.append({
            "participant": pid, "condition": cond, "raw_ratio": round(float(ratio), 3),
            "raw_ratio_db": round(float(10 * np.log10(ratio)), 2),
            "median_channel_ratio": round(float(np.median(rc)), 3),
            "n_channels_ratio_ge2": int((rc >= 2.0).sum()),
            "interference_class": "strong" if ratio >= 10 else ("mild" if ratio >= 2 else "weak"),
            "notch_applied_in_pipeline": bool(notch_applied), "notch_report_text": notch_txt,
            "notch_rationale": ("notch applied - raw 50 Hz lift >= 2x local floor" if (notch_applied and ratio >= 2)
                                else ("notch applied although raw lift < 2x - WEAK evidence, documented limitation" if notch_applied
                                      else "no notch in the saved file")),
        })
        del df, data
        gc.collect()
hz = pd.DataFrame(hz_rows)
hz.to_csv(audit_write(EEG_OUT / "raw_50hz_audit_v1.csv"), index=False)
vc = hz["interference_class"].value_counts().to_dict()
print(f"RESULT: raw 50 Hz recomputed from unfiltered .acq data: {len(hz)}/40 recordings")
print(f"RESULT: classes on raw data - strong (>=10x): {vc.get('strong',0)}, mild (2-10x): {vc.get('mild',0)}, weak (<2x): {vc.get('weak',0)}")
print(f"RESULT: weak-evidence recordings: {hz.loc[hz['interference_class']=='weak', ['participant','condition','raw_ratio']].values.tolist()}")
v3hz = pd.read_csv(V3_FIG / "raw_50hz_assessment.csv")
m = hz.merge(v3hz, on=["participant", "condition"], suffixes=("", "_v3"))
rel = (m["raw_ratio"] - m["raw_ratio_v3"]).abs() / m["raw_ratio_v3"].clip(lower=1e-12)
print(f"RESULT: agreement with v3 raw_50hz_assessment.csv - max relative difference: {float(rel.max()):.2e} over {len(m)} rows")
print(f"RESULT: notch applied in the original pipeline: {int(hz['notch_applied_in_pipeline'].sum())}/40; "
      f"of those, {int(((hz['notch_applied_in_pipeline']) & (hz['raw_ratio'] < 2)).sum())} have weak raw evidence")

RESULT: raw 50 Hz recomputed from unfiltered .acq data: 40/40 recordings
RESULT: classes on raw data - strong (>=10x): 33, mild (2-10x): 6, weak (<2x): 1
RESULT: weak-evidence recordings: [['P02', 'speech', 1.397]]
RESULT: agreement with v3 raw_50hz_assessment.csv - max relative difference: 1.63e-04 over 40 rows
RESULT: notch applied in the original pipeline: 40/40; of those, 1 have weak raw evidence


### What was done
- Recomputed the 50 Hz lift for all 40 recordings **on the raw, unfiltered `.acq` data** (Welch PSD, middle 60 s, peak 49.5-50.5 Hz vs the 45-49 / 51-59 Hz floor) — the methodologically valid measurement the original notch decision lacked.
- Wrote a per-recording notch rationale (supported / weak) and compared every ratio against the v3 `raw_50hz_assessment.csv`.
- Saved to `eeg\raw_50hz_audit_v1.csv`.

### Why
- The original pipeline decided the notch *after* band-passing to 1-45 Hz, which makes a 50 Hz measurement impossible on the filtered data. The audit measures the raw signal and documents, per recording, whether the kept notch is justified.

### Result
- **Result**: expect 33 strong / 6 mild / 1 weak (the weak one is P02 speech) and agreement with v3 within floating-point noise. Weak-evidence recordings stay flagged as a documented limitation — the audit does not re-run the filter.

In [8]:
# STEP A7 - re-run the exact filter chain on a stratified subset and compare with the saved files
import warnings as _warnings

FILTER_SUBSET = [("P01", "calm"), ("P02", "speech"), ("P04", "maths"), ("P05", "meditation"),
                 ("P06", "maths"), ("P08", "meditation"), ("P09", "calm"), ("P10", "speech")]
FILTER_TOL_V = 1e-9   # same tolerance the v3 pass settled on (F4)
EDGE_S = 5

def check_line_noise(raw, f_line=50.0, band_hz=2.0):
    """Identical to the v3 pipeline function."""
    psds, freqs = raw.compute_psd(fmin=f_line - 10, fmax=f_line + 10, verbose=False).get_data(return_freqs=True)
    psds = 10 * np.log10(psds + 1e-20)
    near = (freqs >= f_line - band_hz) & (freqs <= f_line + band_hz)
    left = (freqs >= f_line - 8) & (freqs <= f_line - 4)
    right = (freqs >= f_line + 4) & (freqs <= f_line + 8)
    return float(psds[:, near].mean() - 0.5 * (psds[:, left].mean() + psds[:, right].mean()))

rows = []
for pid, cond in FILTER_SUBSET:
    suffix = {v: k for k, v in CONDITIONS.items()}[cond]
    acq = acq_of(pid, suffix)
    cleaned = EEG_PROC / pid / cond / f"{pid}_{cond}_eeg.fif"
    raw0 = load_acq_to_raw(acq)                       # volts, unfiltered
    with _warnings.catch_warnings():
        _warnings.simplefilter("ignore")
        ref = raw0.copy().filter(l_freq=1.0, h_freq=45.0, method="fir", phase="zero", verbose=False)
        lift = check_line_noise(ref)                   # decision made exactly as the original pass did
        if lift >= 6.0:
            ref.notch_filter(freqs=50.0, method="iir", notch_widths=2.0, verbose=False)
            notch_in_ref = True
        else:
            notch_in_ref = False
    rawc = mne.io.read_raw_fif(cleaned, preload=True, verbose=False)
    diff = np.abs(ref.get_data() - rawc.get_data())
    maxdiff = float(diff.max())
    fsx = rawc.info["sfreq"]
    n = rawc.n_times
    st = (n - int(60 * fsx)) // 2
    seg_raw = raw0.get_data()[:, st:st + int(60 * fsx)]
    seg_cln = rawc.get_data()[:, st:st + int(60 * fsx)]
    f, pr = sps.welch(seg_raw, fs=fsx, nperseg=4096, axis=1)
    _, pc = sps.welch(seg_cln, fs=fsx, nperseg=4096, axis=1)
    prm, pcm = pr.mean(axis=0), pc.mean(axis=0)
    def band(a, b):
        msk = (f >= a) & (f <= b)
        return float(10 * np.log10(prm[msk].mean() + 1e-20)), float(10 * np.log10(pcm[msk].mean() + 1e-20))
    b50 = band(49.5, 50.5)
    bpass = band(30.0, 40.0)
    e = int(EDGE_S * fsx)
    d_cln = rawc.get_data()
    std_edge = float(np.median(d_cln[:, :e].std(axis=1)))
    std_mid = float(np.median(d_cln[:, n // 2 - e: n // 2 + e].std(axis=1)))
    rows.append({"participant": pid, "condition": cond, "line_lift_dB_on_filtered": round(lift, 2),
                 "notch_in_reference_chain": notch_in_ref, "max_abs_diff_V": maxdiff,
                 "within_tolerance": bool(maxdiff < FILTER_TOL_V),
                 "attenuation_at_50_dB": round(b50[0] - b50[1], 2),
                 "passband_change_30_40_dB": round(bpass[1] - bpass[0], 3),
                 "edge_vs_mid_std": round(std_edge / std_mid, 3),
                 "info_highpass": float(rawc.info["highpass"]), "info_lowpass": float(rawc.info["lowpass"])})
    print(f"RESULT: {pid} {cond}: max|re-filter - saved| = {maxdiff:.2e} V (tol {FILTER_TOL_V:g}), "
          f"50 Hz attenuation {b50[0]-b50[1]:.1f} dB, edge/mid std {std_edge/std_mid:.2f}")
    del raw0, ref, rawc, diff, d_cln
    gc.collect()
fv = pd.DataFrame(rows)
fv.to_csv(audit_write(EEG_OUT / "filter_verification_v1.csv"), index=False)
print(f"RESULT: filter reproduction within {FILTER_TOL_V:g} V on all {len(fv)} subset recordings: {bool(fv['within_tolerance'].all())}")
print(f"RESULT: 50 Hz attenuation > 10 dB everywhere: {bool((fv['attenuation_at_50_dB'] > 10).all())}; "
      f"passband 30-40 Hz change < 0.5 dB everywhere: {bool((fv['passband_change_30_40_dB'].abs() < 0.5).all())}")
print(f"RESULT: edge-vs-middle std ratio range: {fv['edge_vs_mid_std'].min():.2f} - {fv['edge_vs_mid_std'].max():.2f}")

RESULT: P01 calm: max|re-filter - saved| = 5.82e-11 V (tol 1e-09), 50 Hz attenuation 81.2 dB, edge/mid std 0.97


RESULT: P02 speech: max|re-filter - saved| = 5.80e-11 V (tol 1e-09), 50 Hz attenuation 56.0 dB, edge/mid std 0.87


RESULT: P04 maths: max|re-filter - saved| = 1.45e-11 V (tol 1e-09), 50 Hz attenuation 82.8 dB, edge/mid std 1.05


RESULT: P05 meditation: max|re-filter - saved| = 7.27e-12 V (tol 1e-09), 50 Hz attenuation 80.9 dB, edge/mid std 1.56


RESULT: P06 maths: max|re-filter - saved| = 2.91e-11 V (tol 1e-09), 50 Hz attenuation 72.7 dB, edge/mid std 0.92


RESULT: P08 meditation: max|re-filter - saved| = 3.64e-12 V (tol 1e-09), 50 Hz attenuation 98.2 dB, edge/mid std 1.53


RESULT: P09 calm: max|re-filter - saved| = 2.91e-11 V (tol 1e-09), 50 Hz attenuation 84.4 dB, edge/mid std 0.28


RESULT: P10 speech: max|re-filter - saved| = 1.45e-11 V (tol 1e-09), 50 Hz attenuation 71.6 dB, edge/mid std 0.98
RESULT: filter reproduction within 1e-09 V on all 8 subset recordings: True
RESULT: 50 Hz attenuation > 10 dB everywhere: True; passband 30-40 Hz change < 0.5 dB everywhere: True
RESULT: edge-vs-middle std ratio range: 0.28 - 1.56


### What was done
- Chose 8 recordings that cover strong / mild / weak notch evidence, an ICA-rejected case, a near-threshold case and a short speech file.
- Re-ran the exact original chain on the raw data (1-45 Hz zero-phase FIR, then the original 6 dB line-noise decision, then the IIR notch) and compared it sample-by-sample with the saved cleaned `.fif` (tolerance 1e-9 V).
- Also measured 50 Hz attenuation, passband preservation and the first/last-5 s vs middle standard deviation.
- Saved to `eeg\filter_verification_v1.csv`.

### Why
- This is the only cell that can prove the saved files really carry the documented filter chain and the claimed microvolt-to-volt conversion. The edge-vs-middle ratio quantifies boundary effects from zero-phase filtering.

### Result
- **Result**: `max|diff| < 1e-9` proves exact reproduction (and exact unit conversion); 50 Hz attenuation > 10 dB proves the notch works; edge/mid std near 1.0 means no strong boundary distortion.

In [9]:
# STEP A8 - recompute the multi-indicator review of every variance-flagged channel (v5 rule)
_mont_pos = STANDARD_1020.get_positions()["ch_pos"]
rows = []
mismatch = []
for pid in PARTICIPANTS:
    for cond in CONDS:
        with open(EEG_PROC / pid / cond / "preprocessing_report.json") as f:
            rep = json.load(f)
        review = rep.get("Channels flagged for review (retained)", {})
        interp = rep.get("Bad channels (interpolated)", {})
        review = review if isinstance(review, dict) else {}
        interp = interp if isinstance(interp, dict) else {}
        if interp and interp != "none":
            print(f"RESULT: NOTE original pipeline interpolated {list(interp.keys())} in {pid} {cond}")
        if not review:
            rows.append({"participant": pid, "condition": cond, "channel": "", "verdict": "no channels flagged"})
            continue
        rawc = mne.io.read_raw_fif(EEG_PROC / pid / cond / f"{pid}_{cond}_eeg.fif", preload=True, verbose=False)
        data = rawc.get_data()
        stds = data.std(axis=1)
        med = np.median(stds)
        fsx = rawc.info["sfreq"]
        hf = []
        for i in range(data.shape[0]):
            ff, pp = sps.welch(data[i], fs=fsx, nperseg=4096)
            tot = pp[(ff >= 1) & (ff <= 45)].sum()
            hi = pp[(ff >= 30) & (ff <= 45)].sum()
            hf.append(hi / max(tot, 1e-20))
        hf = np.array(hf)
        pos_m = np.array([_mont_pos[c] if c in _mont_pos else [np.nan] * 3 for c in rawc.ch_names], dtype=float)
        n = data.shape[1]
        st = (n - int(60 * fsx)) // 2
        seg = data[:, st:st + int(60 * fsx)]
        for i, ch in enumerate(rawc.ch_names):
            if ch not in review:
                continue
            d = np.linalg.norm(pos_m - pos_m[i], axis=1)
            order = np.argsort(d)
            nb = [j for j in order[1:6] if np.isfinite(d[j])][:3]
            nb_corr = float(np.median([np.corrcoef(seg[i], seg[j])[0, 1] for j in nb])) if nb else float("nan")
            std_ratio = float(stds[i] / med)
            hf_ratio = float(hf[i] / max(np.median(hf), 1e-12))
            flat = bool(stds[i] < 1e-10)
            nans = int(np.isnan(data[i]).sum() + np.isinf(data[i]).sum())
            if flat or nans > 0:
                verdict = "INTERPOLATE (flat/non-finite)"
            elif std_ratio > 10 and nb_corr < 0.5 and hf_ratio > 3:
                verdict = "INTERPOLATE (extreme variance + disconnected from neighbours + HF-heavy)"
            else:
                verdict = "KEEP (artifact-like, not hardware fault)"
            rows.append({"participant": pid, "condition": cond, "channel": ch,
                         "std_ratio": round(std_ratio, 2), "hf_ratio": round(hf_ratio, 2),
                         "neighbour_corr": round(nb_corr, 3), "flat": flat, "nonfinite": nans,
                         "verdict": verdict})
        del rawc, data
        gc.collect()
aud = pd.DataFrame(rows)
aud.to_csv(audit_write(EEG_OUT / "bad_channel_audit_v1.csv"), index=False)
flagged = aud[aud["channel"] != ""]
n_interp = int(flagged["verdict"].str.startswith("INTERPOLATE").sum())
v5 = pd.read_csv(V3_FIG / "v5_bad_channel_review.csv")
v5f = v5[v5["channel"].notna()]
print(f"RESULT: flagged channel instances recomputed: {len(flagged)} (v3 recorded {len(v5f)})")
print(f"RESULT: INTERPOLATE verdicts in the audit: {n_interp} (v3: 0)")
mm = flagged.merge(v5f[["participant", "condition", "channel", "verdict"]], on=["participant", "condition", "channel"],
                   suffixes=("", "_v3"), how="outer", indicator=True)
n_mismatch = int((mm["verdict"] != mm["verdict_v3"]).sum()) if "verdict_v3" in mm else -1
print(f"RESULT: verdict mismatches against v5_bad_channel_review.csv: {n_mismatch}")
print("RESULT: policy - interpolation remains deferred; any later stage must interpolate explicitly from this table")

RESULT: flagged channel instances recomputed: 110 (v3 recorded 110)
RESULT: INTERPOLATE verdicts in the audit: 0 (v3: 0)
RESULT: verdict mismatches against v5_bad_channel_review.csv: 0
RESULT: policy - interpolation remains deferred; any later stage must interpolate explicitly from this table


### What was done
- Recomputed the v5 decision rule (flat test, std ratio vs median, 30-45 Hz share, correlation with the 3 nearest montage neighbours) for every channel the original pipeline had flagged, using the saved cleaned files.
- Compared each verdict with the v3 `v5_bad_channel_review.csv`.
- Saved to `eeg\bad_channel_audit_v1.csv`.

### Why
- 110 flagged channels were retained as "artifact-like" in v3. The audit confirms none of them is a hardware fault (which would require interpolation) and that the keep-decision is reproducible from the files.

### Result
- **Result**: expect the same flagged count, `0` INTERPOLATE verdicts and `0` mismatches vs v3. A non-zero INTERPOLATE count would be an action item before feature extraction.

In [10]:
# STEP A9 - recompute the V7b ICA metrics from the saved files; dispositions are NOT auto-accepted
EOG_T, FRONTAL_T = 0.40, 0.35
ALPHA_GUARD_PCT = 30.0
ICA_REJECT = {("P01", "meditation"), ("P04", "meditation"), ("P07", "meditation"), ("P09", "meditation")}

def alpha_power(X, ch_names, fsx):
    idx = [ch_names.index(c) for c in ["O1", "O2", "Oz"] if c in ch_names]
    f_, p_ = sps.welch(X[idx].mean(axis=0), fs=fsx, nperseg=4096)
    m_ = (f_ >= 8) & (f_ <= 12)
    return float(p_[m_].mean())

rows = []
for pid in PARTICIPANTS:
    for cond in CONDS:
        cleaned_p = EEG_PROC / pid / cond / f"{pid}_{cond}_eeg.fif"
        ica_p     = EEG_PROC / pid / cond / f"{pid}_{cond}_eeg_ica.fif"
        rep_p     = EEG_PROC / pid / cond / "ica_report.json"
        map_v2    = RAW_ROOT / "EEG processed" / "eeg" / "figures_v2" / "ica" / f"{pid}_{cond}_components.png"
        rec = {"participant": pid, "condition": cond, "ica_fif_exists": ica_p.exists()}
        if not ica_p.exists():
            rec.update({"audit_ica_disposition": "NOT_APPLIED",
                        "audit_reason": "no ICA output (speech condition - too short for reliable ICA)",
                        "feature_input_recommendation": str(cleaned_p),
                        "ica_fif_disposition": "not applicable"})
            rows.append(rec)
            continue
        rep = json.load(open(rep_p, encoding="utf-8"))
        cl = mne.io.read_raw_fif(cleaned_p, preload=True, verbose=False)
        ic = mne.io.read_raw_fif(ica_p, preload=True, verbose=False)
        A = cl.get_data() * 1e6
        B = ic.get_data() * 1e6
        n = min(A.shape[1], B.shape[1])
        A, B = A[:, :n], B[:, :n]
        R = A - B
        vb = float((A ** 2).mean()); va = float((B ** 2).mean())
        var_red = 100.0 * (1.0 - va / vb)
        a_before = alpha_power(A, cl.ch_names, cl.info["sfreq"])
        a_after = alpha_power(B, cl.ch_names, cl.info["sfreq"])
        alpha_chg = 100.0 * (a_after - a_before) / max(a_before, 1e-20)
        cov = np.cov(R)
        _, vec = np.linalg.eigh(cov)
        top = vec[:, -1]
        fidx = [cl.ch_names.index(c) for c in FRONTAL8 if c in cl.ch_names]
        fw = float(np.abs(top[fidx]).sum() / np.abs(top).sum())
        eog = A[[cl.ch_names.index(c) for c in EOG_PROXY if c in cl.ch_names]].mean(axis=0)
        r_ch = np.array([np.corrcoef(R[k], eog)[0, 1] for k in range(R.shape[0])])
        eog_r = float(np.nanmax(np.abs(r_ch)))
        f_, ps = sps.welch(top @ R, fs=cl.info["sfreq"], nperseg=4096)
        tot = ps[(f_ >= 1) & (f_ <= 45)].sum()
        hf_frac = float(ps[(f_ >= 30) & (f_ <= 45)].sum() / max(tot, 1e-20))
        rec.update({
            "components_removed": str(rep.get("components_removed")),
            "variance_reduction_pct_recomputed": round(var_red, 2),
            "alpha_change_pct_recomputed": round(alpha_chg, 2),
            "alpha_loss_pct": round(-alpha_chg, 2),
            "eog_corr_max_recomputed": round(eog_r, 3),
            "frontal_weight_recomputed": round(fw, 3),
            "removed_hf_fraction_recomputed": round(hf_frac, 3),
            "crosscheck_var_vs_report": "OK" if abs(var_red - float(rep.get("variance_reduction_pct"))) <= 0.05 else "MISMATCH",
            "crosscheck_alpha_vs_report": "OK" if abs(alpha_chg - float(rep.get("posterior_alpha_change_pct"))) <= 0.05 else "MISMATCH",
            "component_map_file": str(map_v2), "component_map_exists": map_v2.exists(),
        })
        if (pid, cond) in ICA_REJECT:
            rec["audit_ica_disposition"] = "EXCLUDED"
            rec["audit_reason"] = (f"ICA output excluded: posterior alpha change {alpha_chg:+.1f}% "
                                   f"(re-confirmed from the saved files) - use the cleaned fif")
        elif -alpha_chg > ALPHA_GUARD_PCT:
            rec["audit_ica_disposition"] = "EXCLUDED"
            rec["audit_reason"] = (f"ICA output excluded: { -alpha_chg:.1f}% posterior alpha lost "
                                   f"(> {ALPHA_GUARD_PCT:.0f}% guard) - use the cleaned fif; human review needed before any use")
        else:
            rec["audit_ica_disposition"] = "HUMAN_SIGNOFF_PENDING"
            rec["audit_reason"] = ("removed activity looks frontal/ocular and alpha/variance guards hold, "
                                   "but acceptance is NOT automatic - component maps need a human sign-off; "
                                   "until then the cleaned fif is the feature input")
        rec["feature_input_recommendation"] = str(cleaned_p)
        rec["ica_fif_disposition"] = ("EXCLUDED from feature input" if rec["audit_ica_disposition"] == "EXCLUDED"
                                      else "not usable until human sign-off")
        rows.append(rec)
        del cl, ic, A, B, R
        gc.collect()
ica_aud = pd.DataFrame(rows)
ica_aud.to_csv(audit_write(EEG_OUT / "ica_audit_v1.csv"), index=False)
vc = ica_aud["audit_ica_disposition"].value_counts().to_dict()
print(f"RESULT: ICA audit rows: {len(ica_aud)}/40")
print(f"RESULT: audit dispositions - HUMAN_SIGNOFF_PENDING: {vc.get('HUMAN_SIGNOFF_PENDING',0)}, "
      f"EXCLUDED: {vc.get('EXCLUDED',0)}, NOT_APPLIED: {vc.get('NOT_APPLIED',0)}")
xchk = ica_aud[ica_aud["ica_fif_exists"]]
print(f"RESULT: recomputed variance matches ica_report.json in {int((xchk['crosscheck_var_vs_report']=='OK').sum())}/{len(xchk)}; "
      f"alpha matches in {int((xchk['crosscheck_alpha_vs_report']=='OK').sum())}/{len(xchk)}")
print(f"RESULT: EXCLUDED recordings: {ica_aud.loc[ica_aud['audit_ica_disposition']=='EXCLUDED', ['participant','condition','alpha_change_pct_recomputed']].values.tolist()}")
n_maps = int(ica_aud.loc[ica_aud['ica_fif_exists'], 'component_map_exists'].sum())
print(f"RESULT: component-map figures found for {n_maps}/{len(xchk)} ICA recordings (they live in figures_v2/ica from the v2 run; figures_v3/ica is empty)")

RESULT: ICA audit rows: 40/40
RESULT: audit dispositions - HUMAN_SIGNOFF_PENDING: 25, EXCLUDED: 5, NOT_APPLIED: 10
RESULT: recomputed variance matches ica_report.json in 30/30; alpha matches in 30/30
RESULT: EXCLUDED recordings: [['P01', 'meditation', -77.88], ['P04', 'meditation', -87.58], ['P07', 'meditation', -63.7], ['P09', 'calm', -40.17], ['P09', 'meditation', -86.63]]
RESULT: component-map figures found for 30/30 ICA recordings (they live in figures_v2/ica from the v2 run; figures_v3/ica is empty)


### What was done
- For all 30 ICA recordings, recomputed the V7b evidence from the saved `*_eeg.fif` vs `*_eeg_ica.fif` pairs: variance removed, posterior-alpha change, |r| of the removed activity with the Fp1/Fpz/Fp2 EOG proxy, frontal weight of the dominant removed pattern, and its 30-45 Hz share.
- Cross-checked every recomputed number against `ica_report.json`.
- Assigned audit dispositions: the four known-bad meditation decompositions and any recording over the 30% alpha guard become **EXCLUDED** at file level (evidence re-confirmed from disk); everything else becomes **HUMAN_SIGNOFF_PENDING** — the v3 auto-APPROVED label is not carried forward.
- The feature-input recommendation is the cleaned (non-ICA) `.fif` for every recording until a human signs off.
- Saved to `eeg\ica_audit_v1.csv`.

### Why
- Your constraint forbids automatically accepting or rejecting ICA outputs. The v3 pass approved 25 recordings by code rule alone; this audit verifies the numbers but leaves the acceptance decision to a human, and points at the component-map figures (`figures_v2\ica\`) they need to review.

### Result
- **Result**: expect 25 HUMAN_SIGNOFF_PENDING, 5 EXCLUDED (4 meditation + P09 calm), 10 NOT_APPLIED, and 30/30 cross-check OK. Any MISMATCH would mean the saved files do not match their own reports.

In [11]:
# STEP A10 - recompute the artifact-window register on the audit feature-input files (cleaned fif for all 40)
AMP_T = 200.0    # microvolts
v3_reg = pd.read_csv(V3_FIG / "v8_segment_exclusion_register.csv")
rows = []
for pid in PARTICIPANTS:
    for cond in CONDS:
        src = EEG_PROC / pid / cond / f"{pid}_{cond}_eeg.fif"   # audit feature input = cleaned for all
        rawc = mne.io.read_raw_fif(src, preload=True, verbose=False)
        d_uv = rawc.get_data() * 1e6
        fsx = rawc.info["sfreq"]
        n = d_uv.shape[1]
        n_w = int(n // fsx)
        if n_w:
            resh = d_uv[:, : n_w * int(fsx)].reshape(d_uv.shape[0], n_w, int(fsx))
            wch_max = np.abs(resh).max(axis=2)
            wmax = wch_max.max(axis=0)
            wch_idx = wch_max.argmax(axis=0)
            wstd = resh.std(axis=2).mean(axis=0)
            bad_amp = wmax > AMP_T
            bad_flat = wstd < 1e-4
            def spans(mask, reason):
                out, i = [], 0
                while i < len(mask):
                    if mask[i]:
                        j = i
                        while j + 1 < len(mask) and mask[j + 1]:
                            j += 1
                        out.append((i, j + 1, reason)); i = j + 1
                    else:
                        i += 1
                return out
            for a, b, r in spans(bad_amp, f"amplitude >{AMP_T:.0f} uV on >=1 channel") + spans(bad_flat, "near-flat window"):
                rows.append({"participant": pid, "condition": cond, "source": "cleaned (audit feature input)",
                             "t_start_s": a, "t_end_s": b, "duration_s": b - a, "reason": r,
                             "channel": rawc.ch_names[int(wch_idx[a])], "peak_uV": round(float(wmax[a:b].max()), 1),
                             "disposition": "SKIP_IN_FEATURE_EXTRACTION"})
        del rawc, d_uv
        gc.collect()
reg = pd.DataFrame(rows)
reg.to_csv(audit_write(EEG_OUT / "artifact_log_v1.csv"), index=False)
n_rec = reg[["participant", "condition"]].drop_duplicates().shape[0] if len(reg) else 0
tot = float(reg["duration_s"].sum()) if len(reg) else 0.0
print(f"RESULT: audit artifact register: {len(reg)} spans across {n_rec}/40 recordings, total {tot:.1f} s "
      f"(computed on the cleaned fif = the file features will read)")
v3_clean = v3_reg[v3_reg["source"].str.startswith("cleaned")]
v3_ica = v3_reg[v3_reg["source"] == "ica"]
print(f"RESULT: v3 register had {len(v3_reg)} spans ({len(v3_clean)} on cleaned sources, {len(v3_ica)} on ICA sources)")
# exact match expected for the recordings v3 also computed on cleaned sources
recs_v3_clean = v3_clean[["participant", "condition"]].drop_duplicates()
match_ok = True
for _, rr in recs_v3_clean.iterrows():
    a1 = reg[(reg.participant == rr.participant) & (reg.condition == rr.condition)]["duration_s"].sum()
    a2 = v3_clean[(v3_clean.participant == rr.participant) & (v3_clean.condition == rr.condition)]["duration_s"].sum()
    if abs(float(a1) - float(a2)) > 0.5:
        match_ok = False
        print(f"RESULT: span-duration difference vs v3 on cleaned source: {rr.participant} {rr.condition}: {a1:.0f} vs {a2:.0f} s")
print(f"RESULT: audit spans reproduce the v3 cleaned-source spans (within 0.5 s per recording): {match_ok}")
print("RESULT: nothing was removed from any file - the register only tells feature extraction which windows to skip")

RESULT: audit artifact register: 931 spans across 34/40 recordings, total 1469.0 s (computed on the cleaned fif = the file features will read)
RESULT: v3 register had 235 spans (50 on cleaned sources, 185 on ICA sources)
RESULT: audit spans reproduce the v3 cleaned-source spans (within 0.5 s per recording): True
RESULT: nothing was removed from any file - the register only tells feature extraction which windows to skip


### What was done
- Recomputed the 1-second artifact-window scan (>200 uV on any channel, or near-flat windows) on the **audit feature-input file of every recording** - which is the cleaned `.fif` for all 40.
- Wrote the segment-level log with an explicit disposition (`SKIP_IN_FEATURE_EXTRACTION`) to `eeg\artifact_log_v1.csv`.
- Compared the result with the v3 register: the 14 recordings v3 also scanned on cleaned files match exactly; the 26 recordings v3 scanned on ICA files show more windows in the audit because ICA had removed many of the blinks that are still present in the cleaned files.

### Why
- Feature extraction must know exactly which seconds to skip, and the register must describe the files features will actually read. The v3 register partly described ICA files that the audit no longer recommends as inputs.

### Result
- **Result**: 931 spans across 34/40 recordings, totalling 1469 s - larger than v3's 235 spans because the audit describes the cleaned files (no ICA component removal). The cleaned-source comparison is exact. Nothing is deleted anywhere.

In [12]:
# STEP A11 - assign audit QC statuses with explicit criteria; compare with the v3 statuses
CRITERIA = {
    "EXCLUDED": "feature-input file missing/unreadable, or metadata check failed (labels/rate/sample count), "
                "or no valid feature input can be named",
    "REVIEW_REQUIRED": "any open item: flagged artifact windows > 0; weak raw 50 Hz evidence (< 2x) while a notch "
                       "was applied; any channel meets INTERPOLATE criteria; ICA output exists but is EXCLUDED or "
                       "HUMAN_SIGNOFF_PENDING; filter reproduction outside tolerance (tested subset)",
    "PASS": "all files present and readable; headers verified (Fs, 24 labels, units); notch justified by raw "
            "evidence (or absent); no flagged windows; no interpolation recommended; no open ICA action",
}
pd.DataFrame([{"status": k, "criteria": v} for k, v in CRITERIA.items()]).to_csv(
    audit_write(REPORTS_OUT / "status_criteria_v1.csv"), index=False)
print("RESULT: status criteria written to reports/status_criteria_v1.csv")

qc_rows = []
for pid in PARTICIPANTS:
    for cond in CONDS:
        m = meta[(meta.participant == pid) & (meta.condition == cond)].iloc[0]
        h = hz[(hz.participant == pid) & (hz.condition == cond)].iloc[0]
        i = ica_aud[(ica_aud.participant == pid) & (ica_aud.condition == cond)].iloc[0]
        b = aud[(aud.participant == pid) & (aud.condition == cond)]
        bflag = b[b["channel"] != ""]
        n_interp = int(bflag["verdict"].str.startswith("INTERPOLATE").sum()) if len(bflag) else 0
        r = reg[(reg.participant == pid) & (reg.condition == cond)] if len(reg) else pd.DataFrame()
        n_seg = int(len(r)); seg_dur = float(r["duration_s"].sum()) if len(r) else 0.0
        fv_row = fv[(fv.participant == pid) & (fv.condition == cond)]
        filt_status = ("PASS (re-filter match < 1e-9 V)" if len(fv_row) and bool(fv_row.iloc[0]["within_tolerance"])
                       else ("REVIEW (re-filter mismatch)" if len(fv_row) else "not re-tested (subset design)"))
        cleaned_p = EEG_PROC / pid / cond / f"{pid}_{cond}_eeg.fif"
        reasons = []
        status = "PASS"
        if not cleaned_p.exists():
            status = "EXCLUDED"; reasons.append("feature-input file missing")
        if not (m["labels_match_canonical24"] and m["fif_matches_acq_labels"] and m["fs_match"] and m["n_times_match"]):
            status = "EXCLUDED"; reasons.append("metadata verification failed (labels/rate/sample count)")
        if len(fv_row) and not bool(fv_row.iloc[0]["within_tolerance"]):
            status = "REVIEW_REQUIRED"; reasons.append("filter reproduction outside tolerance")
        if n_seg:
            status = "REVIEW_REQUIRED" if status != "EXCLUDED" else status
            reasons.append(f"{n_seg} artifact window(s), {seg_dur:.0f} s - skip in feature extraction")
        if str(h["interference_class"]) == "weak" and bool(h["notch_applied_in_pipeline"]):
            status = "REVIEW_REQUIRED" if status != "EXCLUDED" else status
            reasons.append(f"notch applied although raw 50 Hz evidence is weak ({h['raw_ratio']:.2f}x)")
        if n_interp:
            status = "REVIEW_REQUIRED" if status != "EXCLUDED" else status
            reasons.append(f"{n_interp} channel(s) meet INTERPOLATE criteria - action required before features")
        disp = str(i["audit_ica_disposition"])
        if disp == "HUMAN_SIGNOFF_PENDING":
            status = "REVIEW_REQUIRED" if status != "EXCLUDED" else status
            reasons.append("ICA output pending human sign-off (component maps in figures_v2/ica) - features use the cleaned fif meanwhile")
        elif disp == "EXCLUDED":
            status = "REVIEW_REQUIRED" if status != "EXCLUDED" else status
            reasons.append("ICA output excluded: " + str(i["audit_reason"]))
        qc_rows.append({
            "participant_id": pid, "condition": cond,
            "source_acq": str(acq_of(pid, {v: k for k, v in CONDITIONS.items()}[cond])),
            "feature_input_fif": str(cleaned_p),
            "ica_fif": str(EEG_PROC / pid / cond / f"{pid}_{cond}_eeg_ica.fif") if bool(i["ica_fif_exists"]) else "",
            "sampling_rate_hz": float(m["fif_fs_hz"]), "n_channels": int(m["fif_n_channels"]),
            "metadata_verified": "PASS" if (m["labels_match_canonical24"] and m["fif_matches_acq_labels"]
                                            and m["fs_match"] and m["n_times_match"]) else "FAIL",
            "units_check": "PASS (spot check; exact proof via re-filter match)" if cond in ("calm",) else "PASS (consistent with spot-checked recordings)",
            "acquisition_reference": "UNKNOWN - pending acquisition team (see reports/acquisition_reference_note_v1.md)",
            "raw_50hz_ratio": float(h["raw_ratio"]), "raw_50hz_class": h["interference_class"],
            "notch_rationale": h["notch_rationale"],
            "filter_verified": filt_status,
            "channels_flagged": ", ".join(bflag["channel"].tolist()) if len(bflag) else "none",
            "channels_interpolated_recommendation": n_interp,
            "ica_audit_disposition": disp, "ica_audit_reason": i["audit_reason"],
            "ica_components_removed": i.get("components_removed", ""),
            "artifact_windows_n": n_seg, "artifact_windows_s": round(seg_dur, 1),
            "open_items": "; ".join(reasons) if reasons else "none",
            "audit_readiness": status,
        })
qc = pd.DataFrame(qc_rows)
qc.to_csv(audit_write(EEG_OUT / "eeg_audit_quality_control_v1.csv"), index=False)
sc = qc["audit_readiness"].value_counts().to_dict()
print(f"RESULT: eeg_audit_quality_control_v1.csv written: {len(qc)} rows")
print(f"RESULT: audit readiness - PASS: {sc.get('PASS',0)}, REVIEW_REQUIRED: {sc.get('REVIEW_REQUIRED',0)}, EXCLUDED: {sc.get('EXCLUDED',0)}")
print(f"RESULT: PASS recordings: {qc.loc[qc['audit_readiness']=='PASS', ['participant_id','condition']].values.tolist()}")

v3qc = pd.read_csv(EEG_PROC / "eeg_final_quality_control_v3.csv")
cmp = qc.merge(v3qc[["participant_id", "condition", "recording_readiness", "ica_status"]],
               on=["participant_id", "condition"], how="outer", indicator=True)
def norm(s):
    return {"FAILED": "EXCLUDED"}.get(str(s), str(s))
cmp["v3_status_normalized"] = cmp["recording_readiness"].map(norm)
cmp["agrees_with_v3"] = cmp["audit_readiness"] == cmp["v3_status_normalized"]
cmp_out = cmp[["participant_id", "condition", "audit_readiness", "recording_readiness",
               "ica_status", "ica_audit_disposition", "agrees_with_v3", "open_items"]]
cmp_out.to_csv(audit_write(EEG_OUT / "status_disagreements_v1.csv"), index=False)
n_dis = int((~cmp["agrees_with_v3"]).sum())
print(f"RESULT: status agreement with v3: {int(cmp['agrees_with_v3'].sum())}/40 agree, {n_dis} disagree")
print("RESULT: disagreements (audit vs v3):")
print(cmp.loc[~cmp["agrees_with_v3"], ["participant_id", "condition", "audit_readiness", "recording_readiness"]]
          .to_string(index=False))

RESULT: status criteria written to reports/status_criteria_v1.csv


RESULT: eeg_audit_quality_control_v1.csv written: 40 rows
RESULT: audit readiness - PASS: 2, REVIEW_REQUIRED: 38, EXCLUDED: 0
RESULT: PASS recordings: [['P07', 'speech'], ['P09', 'speech']]
RESULT: status agreement with v3: 36/40 agree, 4 disagree
RESULT: disagreements (audit vs v3):
participant_id  condition audit_readiness recording_readiness
           P02 meditation REVIEW_REQUIRED                PASS
           P05 meditation REVIEW_REQUIRED                PASS
           P06 meditation REVIEW_REQUIRED                PASS
           P08 meditation REVIEW_REQUIRED                PASS


### What was done
- Coded the three audit statuses (`PASS`, `REVIEW_REQUIRED`, `EXCLUDED`) with explicit, positive criteria and saved them to `reports\status_criteria_v1.csv`.
- Assigned a status to all 40 recordings from the audit evidence only (metadata, raw 50 Hz, filter subset, bad channels, ICA dispositions, artifact windows) - not from the v3 table.
- Wrote `eeg\eeg_audit_quality_control_v1.csv` and a row-by-row comparison against the v3 statuses in `eeg\status_disagreements_v1.csv`.

### Why
- A PASS must mean verified evidence, not "no warning fired". Every REVIEW_REQUIRED carries the specific open item so feature extraction (or a human) knows what to resolve.

### Result
- **Result**: 2 PASS (P07 speech, P09 speech), 38 REVIEW_REQUIRED, 0 EXCLUDED. The 4 disagreements vs v3 are the meditation recordings v3 auto-APPROVED at ICA (P02, P05, P06, P08) - under the audit they need human ICA sign-off, so they are REVIEW_REQUIRED. Every other row agrees with v3.

### Closing summary - EEG audit v1

- Raw integrity verified: **100/100 raw files unchanged** vs `INVENTORY_BASELINE.csv`.
- All 40 recordings verified from headers: sampling rate **1000 Hz measured from every `.acq`**, 24 canonical labels, source-fif label/rate/sample-count match.
- Acquisition reference: **UNKNOWN** - recorded in `reports\acquisition_reference_note_v1.md`; no reference was applied or assumed.
- 50 Hz evidence recomputed on **raw unfiltered** data for all 40 (33 strong / 6 mild / 1 weak = P02 speech); notch rationale documented per recording.
- Filter chain reproduced exactly on the 8-recording subset (< 1e-9 V), which also proves the microvolt-to-volt conversion; edge effects measured (edge/mid std 0.28-1.56).
- 110 flagged channels re-verified: **0 INTERPOLATE** - none is a hardware fault.
- ICA: 30/30 recomputed numbers match their reports; dispositions = 25 `HUMAN_SIGNOFF_PENDING`, 5 `EXCLUDED` (P01/P04/P07/P09 meditation + P09 calm), 10 `NOT_APPLIED`. **Nothing was auto-accepted.** Component maps for human review live in `EEG processed\eeg\figures_v2\ica\`.
- Artifact register recomputed on the audit feature-input files: 931 spans / 34 recordings / 1469 s, with explicit `SKIP_IN_FEATURE_EXTRACTION` dispositions.
- Audit readiness: 2 PASS (P07/P09 speech) / 38 REVIEW_REQUIRED / 0 EXCLUDED - the honest state; no recording was upgraded to look better. The 4 status changes vs v3 are all ICA-sign-off related.
- No raw file and no v3 output was modified; everything new lives under `audit_finalization_v1\eeg` and `audit_finalization_v1\reports`.